In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"D:/Data_visualization_design/ecommerce-visual-analytics/data/E-Commerce Dataset")


def load_csv_rows(file_name: str, n_rows: int = 5) -> pd.DataFrame:
    """Load the first n_rows from a CSV in the E-Commerce Dataset folder."""
    file_path = DATA_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    return pd.read_csv(file_path, nrows=n_rows)


def list_ecommerce_csv_files() -> list[str]:
    return sorted([p.name for p in DATA_DIR.glob("*.csv")])


In [2]:
df = load_csv_rows("orders_dataset.csv", n_rows=10)
display(df)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [3]:
core_files = [
    "orders_dataset.csv",
    "customers_dataset.csv",
    "order_items_dataset.csv",
    "order_payments_dataset.csv",
    "order_reviews_dataset.csv",
    "products_dataset.csv",
    "sellers_dataset.csv",
]

schema_summary = []

for file_name in core_files:
    file_path = DATA_DIR / file_name
    temp_df = pd.read_csv(file_path, nrows=5)
    schema_summary.append({
        "file": file_name,
        "rows_previewed": len(temp_df),
        "columns": list(temp_df.columns),
    })

schema_summary_df = pd.DataFrame(schema_summary)
display(schema_summary_df)

,file,rows_previewed,columns
0,orders_dataset.csv,5,"[order_id, customer_id, order_status, order_pu..."
1,customers_dataset.csv,5,"[customer_id, customer_unique_id, customer_zip..."
2,order_items_dataset.csv,5,"[order_id, order_item_id, product_id, seller_i..."
3,order_payments_dataset.csv,5,"[order_id, payment_sequential, payment_type, p..."
4,order_reviews_dataset.csv,5,"[review_id, order_id, review_score, review_com..."
5,products_dataset.csv,5,"[product_id, product_category_name, product_na..."
6,sellers_dataset.csv,5,"[seller_id, seller_zip_code_prefix, seller_cit..."


In [4]:
# Build the Master Order Table at one row per order
orders = pd.read_csv(
    DATA_DIR / 'orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
)
customers = pd.read_csv(DATA_DIR / 'customers_dataset.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items_dataset.csv')
order_payments = pd.read_csv(DATA_DIR / 'order_payments_dataset.csv')
order_reviews = pd.read_csv(
    DATA_DIR / 'order_reviews_dataset.csv',
    parse_dates=['review_creation_date', 'review_answer_timestamp'],
)

items_agg = order_items.groupby('order_id', as_index=False).agg(
    order_item_count=('order_item_id', 'count'),
    unique_sellers=('seller_id', 'nunique'),
    unique_products=('product_id', 'nunique'),
    total_freight_value=('freight_value', 'sum'),
    total_item_value=('price', 'sum'),
)

payments_agg = order_payments.groupby('order_id', as_index=False).agg(
    payment_value_total=('payment_value', 'sum'),
    payment_installments_total=('payment_installments', 'sum'),
    payment_type_count=('payment_type', 'nunique'),
)

reviews_agg = (
    order_reviews.sort_values('review_creation_date')
    .drop_duplicates('order_id', keep='last')
    [['order_id', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']]
)

master_order_table = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(items_agg, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_agg, on='order_id', how='left')
)

master_order_table['actual_delivery_days'] = (
    master_order_table['order_delivered_customer_date'] - master_order_table['order_purchase_timestamp']
).dt.days
master_order_table['delivery_delay_days'] = (
    master_order_table['order_delivered_customer_date'] - master_order_table['order_estimated_delivery_date']
).dt.days
master_order_table['late_delivery_flag'] = master_order_table['delivery_delay_days'] > 0
master_order_table['review_risk_flag'] = master_order_table['review_score'] <= 2

output_path = Path.cwd() / 'master_order_table.csv'
master_order_table.to_csv(output_path, index=False)

print(f'Master Order Table saved to: {output_path}')
print(f'Rows: {len(master_order_table):,}')
print(f'Columns: {len(master_order_table.columns):,}')
display(master_order_table.head())

Master Order Table saved to: d:\Data_visualization_design\ecommerce-visual-analytics\data-visualization\master_order_table.csv
Rows: 99,441
Columns: 30


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,actual_delivery_days,delivery_delay_days,late_delivery_flag,review_risk_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,8.0,-8.0,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,-6.0,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,9.0,-18.0,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03,2017-12-05 19:21:58,13.0,-13.0,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51,2.0,-10.0,False,False


In [5]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

display(master_order_table.head())
print(master_order_table.columns.tolist())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_count,unique_sellers,unique_products,total_freight_value,total_item_value,payment_value_total,payment_installments_total,payment_type_count,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,actual_delivery_days,delivery_delay_days,late_delivery_flag,review_risk_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,8.72,29.99,38.71,3.0,2.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,22.76,118.70,141.46,1.0,1.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,-6.0,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,19.22,159.90,179.12,3.0,1.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,9.0,-18.0,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,27.20,45.00,72.20,1.0,1.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,2017-12-03,2017-12-05 19:21:58,13.0,-13.0,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,8.72,19.90,28.62,1.0,1.0,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51,2.0,-10.0,False,False


['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_count', 'unique_sellers', 'unique_products', 'total_freight_value', 'total_item_value', 'payment_value_total', 'payment_installments_total', 'payment_type_count', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'actual_delivery_days', 'delivery_delay_days', 'late_delivery_flag', 'review_risk_flag']
